<a href="https://colab.research.google.com/github/ileeee-sh/SeSAC_29CM_Project/blob/main/PyKoSpacing_x_SOYNLP_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 29CM 앱 리뷰 한국어 전처리 파이프라인
1. **PyKoSpacing** - 띄어쓰기 보정
2. **SOYNLP emoticon_normalize** - 반복 이모티콘 정규화 (ㅋㅋㅋㅋ → ㅋㅋ)
3. **SOYNLP LTokenizer** - 토큰화
4. **emoji 라이브러리** - 이모지 별도 추출

In [ ]:
!pip install git+https://github.com/haven-jeon/PyKoSpacing.git --quiet
!pip install soynlp emoji --quiet

In [ ]:
import pandas as pd
import emoji
from pykospacing import Spacing
from soynlp.normalizer import emoticon_normalize
from soynlp.tokenizer import LTokenizer

In [ ]:
from google.colab import files

uploaded = files.upload()  # 파일 업로드 창 실행
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)
print(f'파일 로드 완료: {df.shape[0]}행 × {df.shape[1]}열')
print(f'컬럼 목록: {list(df.columns)}')
df.head()

In [ ]:
# PyKoSpacing 초기화
spacing = Spacing()

In [ ]:
def extract_emojis(text: str) -> list:
    """이모지만 추출하여 리스트로 반환"""
    return [char for char in text if char in emoji.EMOJI_DATA]


def remove_emojis(text: str) -> str:
    """텍스트에서 이모지 제거 (PyKoSpacing/SOYNLP 처리 전 분리용)"""
    return ''.join(char for char in text if char not in emoji.EMOJI_DATA)


def preprocess_review(text: str, spacing_model, tokenizer) -> dict:
    """
    단일 리뷰 텍스트 전처리 파이프라인

    Returns:
        dict: {
            'emojis'          : 원문에서 추출한 이모지 리스트
            'spaced'          : PyKoSpacing 띄어쓰기 보정 결과
            'normalized'      : emoticon_normalize 결과
            'tokens'          : LTokenizer 토큰 리스트
        }
    """
    if not isinstance(text, str) or text.strip() == '':
        return {
            'emojis': [],
            'spaced': text,
            'normalized': text,
            'tokens': []
        }

    # Step 1: 이모지 추출 (원문 기준)
    emojis = extract_emojis(text)

    # Step 2: 이모지 제거 후 PyKoSpacing 띄어쓰기 보정
    text_no_emoji = remove_emojis(text).strip()
    try:
        spaced = spacing_model(text_no_emoji) if text_no_emoji else text_no_emoji
    except Exception:
        spaced = text_no_emoji

    # Step 3: SOYNLP emoticon_normalize (반복 이모티콘 축약, 이모지는 이미 제거됨)
    normalized = emoticon_normalize(spaced, num_repeats=2)

    # Step 4: LTokenizer 토큰화
    tokens = tokenizer.tokenize(normalized, flatten=True)

    return {
        'emojis': emojis,
        'spaced': spaced,
        'normalized': normalized,
        'tokens': tokens
    }

In [ ]:
from soynlp.word import WordExtractor

# content 컬럼의 텍스트를 문장 리스트로 준비
sentences = df['content'].dropna().astype(str).tolist()

# WordExtractor로 단어 점수 학습
word_extractor = WordExtractor(min_frequency=5)
word_extractor.train(sentences)
words = word_extractor.extract()

# LTokenizer 초기화 (cohesion 점수 사용)
cohesion_scores = {word: score.cohesion_forward for word, score in words.items()}
tokenizer = LTokenizer(scores=cohesion_scores)

print(f'추출된 단어 수: {len(cohesion_scores):,}')

In [ ]:
# Sample Testing
sample = df['content'].dropna().iloc[10]
print(f'[원문]\n{sample}\n')

result = preprocess_review(sample, spacing, tokenizer)
print(f'추출 이모지: {result["emojis"]}')
print(f'띄어쓰기 보정: {result["spaced"]}')
print(f'이모티콘 정규화: {result["normalized"]}')
print(f'토큰 리스트: {result["tokens"]}')

In [ ]:
from tqdm.notebook import tqdm
tqdm.pandas()

# 전처리 실행
results = df['content'].progress_apply(
    lambda x: preprocess_review(x, spacing, tokenizer)
)

# 결과를 새 컬럼으로 추가
df['emojis']     = results.apply(lambda x: x['emojis'])
df['spaced']     = results.apply(lambda x: x['spaced'])
df['normalized'] = results.apply(lambda x: x['normalized'])
df['tokens']     = results.apply(lambda x: x['tokens'])

print('전처리 완료!')
df[['content', 'spaced', 'normalized', 'tokens', 'emojis']].head(10)

In [ ]:
output_path = '29cm_reviews_preprocessed.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')
files.download(output_path)